# 02 — Feature Engineering (PySpark)
**SF Crime Classification | Databricks Pipeline — Step 2 of 4**

> Prerequisite: Run `01_ingestion_delta.ipynb` first.


In [ ]:
from pyspark.sql import functions as F
import math

CATALOG = "workspace"
SCHEMA  = "default"

train = spark.read.table(f"{CATALOG}.{SCHEMA}.train_raw")
test  = spark.read.table(f"{CATALOG}.{SCHEMA}.test_raw")
print(f"Train: {train.count():,} | Test: {test.count():,}")


In [ ]:
def add_time_features(df):
    PI2 = 2 * math.pi
    df = (df
        .withColumn("Hour",        F.hour("Dates"))
        .withColumn("Minute",      F.minute("Dates"))
        .withColumn("Month",       F.month("Dates"))
        .withColumn("Year",        F.year("Dates"))
        .withColumn("DayOfMonth",  F.dayofmonth("Dates"))
        .withColumn("DayOfYear",   F.dayofyear("Dates"))
        .withColumn("WeekOfYear",  F.weekofyear("Dates"))
        .withColumn("Quarter",     F.quarter("Dates"))
        .withColumn("WeekdayNum",  F.dayofweek("Dates"))
    )
    df = df.withColumn("Season",
        F.when(F.col("Month").isin(12,1,2), 0)
         .when(F.col("Month").isin(3,4,5),  1)
         .when(F.col("Month").isin(6,7,8),  2)
         .otherwise(3))
    df = (df
        .withColumn("IsWeekend",       F.col("WeekdayNum").isin(1,7).cast("int"))
        .withColumn("IsNight",         ((F.col("Hour") >= 22) | (F.col("Hour") <= 5)).cast("int"))
        .withColumn("IsLateNight",     F.col("Hour").between(0, 3).cast("int"))
        .withColumn("IsMorning",       F.col("Hour").between(6, 11).cast("int"))
        .withColumn("IsAfternoon",     F.col("Hour").between(12, 17).cast("int"))
        .withColumn("IsEvening",       F.col("Hour").between(18, 21).cast("int"))
        .withColumn("IsRushHour",      (F.col("Hour").between(7,9) | F.col("Hour").between(16,19)).cast("int"))
        .withColumn("IsBusinessHours", (F.col("Hour").between(9,17) & (F.col("IsWeekend") == 0)).cast("int"))
        .withColumn("HourSin",  F.sin(F.col("Hour")       * PI2 / 24))
        .withColumn("HourCos",  F.cos(F.col("Hour")       * PI2 / 24))
        .withColumn("MonthSin", F.sin(F.col("Month")      * PI2 / 12))
        .withColumn("MonthCos", F.cos(F.col("Month")      * PI2 / 12))
        .withColumn("DowSin",   F.sin(F.col("WeekdayNum") * PI2 / 7))
        .withColumn("DowCos",   F.cos(F.col("WeekdayNum") * PI2 / 7))
        .withColumn("DaySin",   F.sin(F.col("DayOfYear")  * PI2 / 365))
        .withColumn("DayCos",   F.cos(F.col("DayOfYear")  * PI2 / 365))
    )
    return df

train = add_time_features(train)
test  = add_time_features(test)
print("✅ Time features added")


In [ ]:
def add_address_features(df):
    df = (df
        .withColumn("IsBlock",        F.col("Address").contains("BLOCK").cast("int"))
        .withColumn("IsIntersection", F.col("Address").contains("/").cast("int"))
        .withColumn("StreetNum",
            F.when(F.regexp_extract("Address", r"^(\d+)", 1) != "",
                   F.regexp_extract("Address", r"^(\d+)", 1).cast("int"))
             .otherwise(0))
    )
    return df.withColumn("StreetNumBin",
        F.when(F.col("StreetNum") == 0,   0)
         .when(F.col("StreetNum") <= 500,  1)
         .when(F.col("StreetNum") <= 1000, 2)
         .when(F.col("StreetNum") <= 2000, 3)
         .when(F.col("StreetNum") <= 5000, 4)
         .otherwise(5))

train = add_address_features(train)
test  = add_address_features(test)
print("✅ Address features added")


In [ ]:
def add_spatial_features(df):
    CX, CY = -122.4074, 37.7849
    return (df
        .withColumn("X_round2",   F.round("X", 2))
        .withColumn("Y_round2",   F.round("Y", 2))
        .withColumn("X_round3",   F.round("X", 3))
        .withColumn("Y_round3",   F.round("Y", 3))
        .withColumn("DistCenter", F.sqrt((F.col("X")-CX)**2 + (F.col("Y")-CY)**2))
        .withColumn("GridX",      ((F.col("X") - (-122.55)) / 0.01).cast("int"))
        .withColumn("GridY",      ((F.col("Y") - 37.65)     / 0.01).cast("int"))
    )

train = add_spatial_features(train)
test  = add_spatial_features(test)
print("✅ Spatial features added")


In [ ]:
coord_freq = train.groupBy("X_round2","Y_round2").agg(F.count("*").alias("CoordFreq"))
train = train.join(coord_freq, on=["X_round2","Y_round2"], how="left")
test  = test.join(coord_freq,  on=["X_round2","Y_round2"], how="left").fillna({"CoordFreq": 0})
print("✅ CoordFreq added")


In [ ]:
# ── Categorical encoding (no MLlib — uses create_map) ─────
from pyspark.sql.functions import create_map, lit
from itertools import chain

def make_index_map(df, col):
    """Build a string→int mapping from distinct values (sorted for reproducibility)."""
    vals = sorted([r[col] for r in df.select(col).distinct().collect()])
    return {v: i for i, v in enumerate(vals)}

district_map = make_index_map(train, "PdDistrict")
day_map      = make_index_map(train, "DayOfWeek")

def apply_map(df, col, mapping, out_col):
    expr = create_map([lit(x) for x in chain(*mapping.items())])
    return df.withColumn(out_col, expr[df[col]].cast("double")).fillna({out_col: len(mapping)})

train = apply_map(train, "PdDistrict", district_map, "District_enc")
train = apply_map(train, "DayOfWeek",  day_map,      "DayOfWeek_enc")
test  = apply_map(test,  "PdDistrict", district_map, "District_enc")
test  = apply_map(test,  "DayOfWeek",  day_map,      "DayOfWeek_enc")

# District frequency rate
dist_total = train.count()
dist_freq  = train.groupBy("PdDistrict").agg((F.count("*") / dist_total).alias("DistrictRate"))
train = train.join(dist_freq, on="PdDistrict", how="left")
test  = test.join(dist_freq,  on="PdDistrict", how="left").fillna({"DistrictRate": 0.0})

print(f"✅ District values: {len(district_map)}")
print(f"✅ DayOfWeek values: {len(day_map)}")


In [ ]:
def add_interactions(df):
    return (df
        .withColumn("District_Hour",      F.col("District_enc") * 100 + F.col("Hour"))
        .withColumn("District_DayOfWeek", F.col("District_enc") * 10  + F.col("DayOfWeek_enc"))
        .withColumn("Hour_DayOfWeek",     F.col("Hour")          * 10  + F.col("DayOfWeek_enc"))
        .withColumn("District_IsNight",   F.col("District_enc") * 2   + F.col("IsNight"))
        .withColumn("District_IsWeekend", F.col("District_enc") * 2   + F.col("IsWeekend"))
    )

train = add_interactions(train)
test  = add_interactions(test)
print("✅ Interaction features added")


In [ ]:
# ── Target label encoding (Category → integer) ────────────
category_map = make_index_map(train, "Category")
label_map    = {v: k for k, v in category_map.items()}   # int → category name

train = apply_map(train, "Category", category_map, "Target")

print(f"✅ Target encoded — {len(category_map)} crime classes")
print("Classes:", list(category_map.keys())[:5], "...")


In [ ]:
FEATURES = [
    "Hour","Minute","Month","Year","DayOfMonth","DayOfYear","WeekOfYear","Quarter","Season",
    "IsWeekend","IsNight","IsLateNight","IsMorning","IsAfternoon","IsEvening","IsRushHour","IsBusinessHours",
    "HourSin","HourCos","MonthSin","MonthCos","DowSin","DowCos","DaySin","DayCos",
    "X","Y","DistCenter","CoordFreq","X_round2","Y_round2","X_round3","Y_round3","GridX","GridY",
    "IsBlock","IsIntersection","StreetNum","StreetNumBin",
    "District_enc","DayOfWeek_enc","DistrictRate",
    "District_Hour","District_DayOfWeek","Hour_DayOfWeek","District_IsNight","District_IsWeekend",
]
print(f"Total features: {len(FEATURES)}")


In [ ]:
(train.select(FEATURES + ["Target"])
      .write.format("delta").mode("overwrite").option("overwriteSchema","true")
      .saveAsTable(f"{CATALOG}.{SCHEMA}.features_train"))

(test.select(FEATURES + ["Id"])
     .write.format("delta").mode("overwrite").option("overwriteSchema","true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.features_test"))

print(f"✅ {CATALOG}.{SCHEMA}.features_train")
print(f"✅ {CATALOG}.{SCHEMA}.features_test")


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([StructField("label_idx", IntegerType()), StructField("category", StringType())])
label_df = spark.createDataFrame([(int(k), v) for k, v in label_map.items()], schema=schema)
(label_df.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.label_map"))

print(f"✅ {CATALOG}.{SCHEMA}.label_map")


---
**Next:** Run `03_model_training.ipynb`.